<a href="https://colab.research.google.com/github/Basil-Maqbool/flyrank-internship-assignment1/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Basil-Maqbool/flyrank-internship-assignment1/blob/main/work/notebooks/w03_data_contract.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

1.What one row means: In our fact table, one row represents the performance of a single unique content page (URL) for a specific client on a single specific day.

2. Tables used: dim_content for page metadata, joined with fact_content_daily_performance for the time-series metrics.

3. Time window: I am iterating on a mid-panel month: March 2026 (2026-03). I am deliberately avoiding the _sample table (June 2026) to keep the final month sealed as a pure test set.

4. Target/Proxy: I am predicting is_declining_label (where trend_direction is "down").

5. Deliberately Excluded: I am completely excluding product-generated decision flags (like health_score, priority_score, or action_type) to ensure the model learns from raw evidence, not from the existing FlyRank system's rules.

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

**Feature** (knowable BEFORE prediction, safe to use):
- `content_age_days` — static metadata, always available
- `days_since_last_update` — static metadata, always available
- `impressions_90d` — trailing 90-day aggregate, ends at export time
- `word_count` — current observable length
- `clicks_90d`, `sessions_90d`, `pageviews_90d` — trailing 90-day aggregates
- `engagement_rate`, `ctr` — derived rates from trailing 90-day data
- `search_volume`, `competition`, `cpc` — keyword metadata
- `content_type`, `main_intent` — categorical metadata

**Label / proxy** (the thing you predict — NEVER a feature):
- `trend_direction` — derived from trend_pct; IS the label source
- `trend_pct` — computed from impressions windows; label source
- `is_declining_label` — derived column: 1 when trend_direction == 'down'

**Context** (for grouping, joining, splitting — never for the model to learn from):
- `content_id` — pseudonymous page ID; grouping/joins only
- `client_id` — pseudonymous client ID; use for grouped train/test splits

**Excluded** (private, product-decision, or non-predictive — each gets a why):
- `provider_used` — LLM provider; not a predictive signal for decay
- `model_used` — LLM model name; not a predictive signal for decay
- `avg_position` — contains the '0 means no data' trap (1,205 rows); requires careful handling

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [1]:
%pip install -U duckdb --quiet

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import duckdb
import os
from dotenv import load_dotenv
load_dotenv()

print(duckdb.__version__)

con = duckdb.connect()

con.execute("INSTALL httpfs;")
con.execute("LOAD httpfs;")

hf_token = os.environ.get('HF_TOKEN')
con.execute(f"CREATE SECRET (TYPE HUGGINGFACE, TOKEN '{hf_token}');")

print("--- FACT 1: Row Count and Date Span ---")
q1 = """
SELECT COUNT(*) as row_count, MIN(report_date) as min_date, MAX(report_date) as max_date
FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
"""
display(con.execute(q1).df())

print("\n--- FACT 2: The Grain (Is it really Day x Client x Content?) ---")
q2 = """
SELECT
    COUNT(*) as total_rows,
    COUNT(DISTINCT report_date || client_hash_id || content_hash_id) as unique_grain_count
FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
"""
display(con.execute(q2).df())

print("\n--- FACT 3: Availability (Filtering with IS TRUE) ---")
q3 = """
SELECT COUNT(*) as rows_with_analytics
FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
WHERE ga4_data_available IS TRUE
"""
display(con.execute(q3).df())

1.5.5
--- FACT 1: Row Count and Date Span ---


,row_count,min_date,max_date
0,9841378,2026-03-01,2026-03-31



--- FACT 2: The Grain (Is it really Day x Client x Content?) ---


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,unique_grain_count
0,9841378,9841378



--- FACT 3: Availability (Filtering with IS TRUE) ---


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,rows_with_analytics
0,413966


In [3]:
# Missingness queries per column — the writing-data-contracts skill requires this
# Use the starter CSV since we're checking the 30k-row dataset

import pandas as pd

df_local = pd.read_csv('../../data/raw/content_refresh_anonymized.csv')

print("=== MISSINGNESS PER COLUMN ===")
missing_report = pd.DataFrame({
    'column': df_local.columns,
    'missing_count': df_local.isna().sum().values,
    'missing_pct': (df_local.isna().mean().values * 100).round(1)
}).sort_values('missing_pct', ascending=False)

# Only show columns with missing data
missing_nonzero = missing_report[missing_report['missing_count'] > 0]
display(missing_nonzero)

print(f"\nTotal columns: {len(df_local.columns)}")
print(f"Columns with missing data: {len(missing_nonzero)}")

=== MISSINGNESS PER COLUMN ===


,column,missing_count,missing_pct
10,provider_used,21438,71.5
8,word_count,7699,25.7
9,char_count,7699,25.7
33,word_count_tier,7699,25.7
34,char_count_tier,7699,25.7
11,model_used,5733,19.1
43,trend_pct,3388,11.3
4,competition_level,2610,8.7
2,search_volume,2468,8.2
5,cpc,2468,8.2



Total columns: 44
Columns with missing data: 13


In [4]:
# Missingness by content_type — the skill says to check if missingness follows categories

print("=== MISSINGNESS BY CONTENT TYPE ===")
print("(A blind fillna(0) injects category signal — check this first)")
print()

key_cols = ['word_count', 'search_volume', 'competition', 'main_intent']
for col in key_cols:
    ct_missing = df_local.groupby('content_type')[col].apply(lambda x: x.isna().mean() * 100).round(1)
    print(f"{col} missing % by content_type:")
    print(ct_missing.to_string())
    print()

=== MISSINGNESS BY CONTENT TYPE ===
(A blind fillna(0) injects category signal — check this first)

word_count missing % by content_type:
content_type
comparison article     0.0
feedly article         0.0
keyword article       28.3

search_volume missing % by content_type:
content_type
comparison article      0.0
feedly article        100.0
keyword article         1.4

competition missing % by content_type:
content_type
comparison article      0.0
feedly article        100.0
keyword article         1.4

main_intent missing % by content_type:
content_type
comparison article      0.0
feedly article        100.0
keyword article         1.0



The Leakage Trap Experiment: To demonstrate leakage, I intentionally added trend_pct to my feature set. Because our target label (is_declining_label) is mathematically derived from the trend calculation, trend_pct is simply the answer in disguise.

When I included it, the model's Precision@50 spiked to an unrealistic near-perfect score. The tree simply split on trend_pct < 0 and ignored all actual SEO signals. I have now deleted trend_pct from the feature frame to keep the model honest.

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

Limitation of this slice: By filtering our analysis to rows where ga4_data_available IS TRUE, we are creating a blind spot. We are entirely dropping pages or clients that do not have Google Analytics tracking properly configured, meaning our model will not be able to score or prioritize a segment of the client's total web presence.